In [ ]:
import os
import gc
import re
import sys
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    TRAIN_FILE_PUB = "/kaggle/input/ieltsdata/public_train.csv"
    TRAIN_FILE_PRIV = "/kaggle/input/ieltsdata/private_train.csv" # Assuming it's in the same dir
    
    # Target Test File
    TEST_FILE = "/kaggle/input/ieltsdata/private_test.csv"
    
    # Outputs
    TABULAR_TRAIN_OUT = "tabular_train_max_features.csv"
    TABULAR_TEST_OUT = "tabular_test_max_features.csv"
    
    # Model
    MODEL_NAME = "microsoft/deberta-v3-large" 
    MAX_LEN = 640
    FOLDS = 5
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. MASSIVE FEATURE ENGINEERING (PURE PYTHON)
# ==========================================
def extract_features(df):
    print(f"⚙️ Extracting 35+ Linguistic Features for {len(df)} rows...")
    df = df.copy()
    
    # --- Helpers ---
    def clean(text): return str(text).strip()
    def get_words(text): return re.findall(r"\w+(?:'\w+)?", clean(text).lower())
    def get_sentences(text): return re.split(r'[.!?]+', clean(text))
    
    # Syllable Counter (Heuristic)
    def count_syllables(word):
        word = word.lower()
        if len(word) <= 3: return 1
        count = len(re.findall(r'[aeiouy]+', word))
        if word.endswith('e'): count -= 1
        if word.endswith('le') and len(word) > 2 and word[-3] not in 'aeiouy': count += 1
        return max(1, count)

    # --- Feature Functions ---
    
    # 1. READABILITY FORMULAS
    def readability_stats(row):
        text = clean(row['Essay'])
        words = get_words(text)
        sentences = [s for s in get_sentences(text) if len(s.split()) > 0]
        
        n_words = len(words)
        n_sentences = len(sentences)
        n_syllables = sum(count_syllables(w) for w in words)
        n_complex = sum(1 for w in words if count_syllables(w) >= 3)
        n_chars = len(text)
        
        if n_words == 0 or n_sentences == 0:
            return pd.Series([0, 0, 0, 0, 0])
        
        # Averages
        avg_sent_len = n_words / n_sentences
        avg_word_len = n_chars / n_words
        avg_syll_word = n_syllables / n_words
        
        # Formulas
        flesch_ease = 206.835 - (1.015 * avg_sent_len) - (84.6 * avg_syll_word)
        gunning_fog = 0.4 * (avg_sent_len + 100 * (n_complex / n_words))
        smog = 1.0430 * math.sqrt(n_complex * (30 / n_sentences)) + 3.1291 if n_sentences >= 30 else 0
        
        return pd.Series([flesch_ease, gunning_fog, smog, avg_sent_len, n_complex])

    # 2. VOCABULARY RICHNESS
    def lexical_stats(row):
        words = get_words(row['Essay'])
        if len(words) == 0: return pd.Series([0, 0, 0])
        
        unique_words = set(words)
        ttr = len(unique_words) / len(words) # Type-Token Ratio
        
        # Words longer than 6 chars (proxy for "sophisticated" vocabulary)
        long_words = sum(1 for w in words if len(w) > 6)
        long_word_ratio = long_words / len(words)
        
        return pd.Series([len(unique_words), ttr, long_word_ratio])

    # 3. ERROR HEURISTICS (Grammar Proxies)
    def error_proxies(row):
        text = clean(row['Essay'])
        
        # Double spaces (typo)
        double_spaces = len(re.findall(r'  ', text))
        
        # Space before punctuation (e.g., "word ." instead of "word.")
        space_punct = len(re.findall(r' \.', text)) + len(re.findall(r' ,', text))
        
        # Sentence starting with lowercase
        sentences = [s.strip() for s in get_sentences(text) if s.strip()]
        lower_starts = sum(1 for s in sentences if len(s) > 0 and s[0].islower())
        
        # Repeated words ("the the")
        repeated = len(re.findall(r'\b(\w+)\s+\1\b', text.lower()))
        
        return pd.Series([double_spaces, space_punct, lower_starts, repeated])

    # 4. PROMPT RELEVANCE
    def prompt_overlap(row):
        p_words = set(get_words(row.get('Prompt', '')))
        e_words = set(get_words(row.get('Essay', '')))
        # Remove common stopwords to find meaningful overlap
        stopwords = {'the','and','is','in','at','of','a','an','to','for','with','on'}
        p_words -= stopwords
        
        overlap = len(p_words.intersection(e_words))
        ratio = overlap / len(p_words) if len(p_words) > 0 else 0
        return pd.Series([overlap, ratio])

    # --- APPLYING FEATURES ---
    
    # Basic Counts
    df['char_count'] = df['Essay'].apply(lambda x: len(clean(x)))
    df['word_count'] = df['Essay'].apply(lambda x: len(get_words(x)))
    df['sentence_count'] = df['Essay'].apply(lambda x: len([s for s in get_sentences(x) if len(s.split())>0]))
    df['paragraph_count'] = df['Essay'].apply(lambda x: len(re.findall(r'\n+', clean(x))) + 1)
    
    # Structure
    df['question_marks'] = df['Essay'].apply(lambda x: str(x).count('?'))
    df['exclamation_marks'] = df['Essay'].apply(lambda x: str(x).count('!'))
    df['semicolons'] = df['Essay'].apply(lambda x: str(x).count(';'))
    df['quotes'] = df['Essay'].apply(lambda x: str(x).count('"'))
    
    # Advanced Blocks
    df[['flesch_ease', 'gunning_fog', 'smog', 'avg_sent_len', 'n_complex']] = df.apply(readability_stats, axis=1)
    df[['unique_words', 'ttr', 'long_word_ratio']] = df.apply(lexical_stats, axis=1)
    df[['err_double_space', 'err_space_punct', 'err_lower_start', 'err_repeated']] = df.apply(error_proxies, axis=1)
    df[['prompt_overlap_count', 'prompt_overlap_ratio']] = df.apply(prompt_overlap, axis=1)
    
    return df

# ==========================================
# 3. MODEL ARCHITECTURE (DEBERTA)
# ==========================================
class AttentionPooling(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(in_dim, in_dim), nn.LayerNorm(in_dim), nn.GELU(), nn.Linear(in_dim, 1))
    def forward(self, last_hidden_state, attention_mask):
        w = self.attention(last_hidden_state).float()
        w[attention_mask==0] = float('-inf')
        w = torch.softmax(w, 1)
        return torch.sum(w * last_hidden_state, dim=1)

class IELTS_Model(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.base_model = AutoModel.from_pretrained(model_name, config=self.config)
        self.attention_pool = AttentionPooling(self.config.hidden_size)
        self.fc = nn.Linear(self.config.hidden_size, 16) 
    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.fc(self.attention_pool(outputs.last_hidden_state, attention_mask))

class IELTSOrdinalDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer
    def __len__(self): return len(self.df)
    def __getitem__(self, index):
        row = self.df.iloc[index]
        prompt = str(row['Prompt']).strip()
        essay = str(row['Essay']).strip()
        img_desc = str(row.get('Image Description', '')).strip()
        if img_desc.lower() == 'nan': img_desc = ""
        full_text = f"PROMPT: {prompt} {self.tokenizer.sep_token} IMAGE: {img_desc} {self.tokenizer.sep_token} ESSAY: {essay}" if img_desc else f"PROMPT: {prompt} {self.tokenizer.sep_token} ESSAY: {essay}"
        inputs = self.tokenizer.encode_plus(full_text, add_special_tokens=True, truncation=True, max_length=Config.MAX_LEN, return_attention_mask=True, return_tensors='pt')
        return {'input_ids': inputs['input_ids'].flatten(), 'attention_mask': inputs['attention_mask'].flatten()}

class SmartCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, batch):
        input_ids = [item['input_ids'] for item in batch]
        batch_inputs = self.tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
        return {'input_ids': batch_inputs['input_ids'], 'attention_mask': batch_inputs['attention_mask']}

# ==========================================
# 4. EXECUTION
# ==========================================
def predict_fn(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            input_ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
            logits = model(input_ids, mask)
            probs = torch.sigmoid(logits)
            pred_scores = 1.0 + (probs.sum(dim=1) * 0.5)
            preds.extend(pred_scores.cpu().numpy())
    return np.array(preds)

def main():
    print(f"🔹 Device: {Config.DEVICE}")
    
    # ----------------------------------------------------
    # 1. LOAD AND MERGE DATA (Fixed Logic)
    # ----------------------------------------------------
    print("📂 Loading & Merging Datasets...")
    
    # Check if files exist, else try standard names
    if not os.path.exists(Config.TRAIN_FILE_PUB):
        # Fallback if specific file path is wrong
        Config.TRAIN_FILE_PUB = "/kaggle/input/ieltsdata/public_train.csv"
        Config.TRAIN_FILE_PRIV = "/kaggle/input/ieltsdata/private_train.csv"
        
    df_pub = pd.read_csv(Config.TRAIN_FILE_PUB)
    df_priv = pd.read_csv(Config.TRAIN_FILE_PRIV)
    df_test = pd.read_csv(Config.TEST_FILE)
    
    # Standardize Column Names
    for df in [df_pub, df_priv, df_test]:
        df.columns = df.columns.str.strip()
        df.rename(columns={'prompt': 'Prompt', 'essay': 'Essay', 'Overall Score': 'Overall', 'image description': 'Image Description'}, inplace=True)
    
    # MERGE (Axis=0 means adding rows)
    df_train = pd.concat([df_pub, df_priv], axis=0).reset_index(drop=True)
    df_train.dropna(subset=['Prompt', 'Essay', 'Overall'], inplace=True)
    
    print(f"✅ Public Train: {len(df_pub)} | Private Train: {len(df_priv)}")
    print(f"✅ FINAL MERGED TRAIN: {len(df_train)} rows")
    
    # ----------------------------------------------------
    # 2. GENERATE PREDICTIONS
    # ----------------------------------------------------
    # Init Predictions
    df_train['deberta_pred'] = 0.0
    test_accum = []
    
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    collator = SmartCollator(tokenizer)
    
    skf = StratifiedKFold(n_splits=Config.FOLDS, shuffle=True, random_state=42)
    y_stratify = df_train['Overall'].astype(str)
    
    print("\n🔮 Generating NLP Predictions (OOF + Test)...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, y_stratify)):
        model_path = f"/kaggle/input/large-sigma-private/model_fold_{fold}.pth"
        
        if not os.path.exists(model_path): 
            # If standard fold model is missing, check alternative path or skip
            alt_path = f"/kaggle/input/best-model/pytorch/default/2/model_fold_{fold}.pth"
            if os.path.exists(alt_path):
                model_path = alt_path
            else:
                print(f"❌ {model_path} missing! Skipping fold."); continue
            
        model = IELTS_Model(Config.MODEL_NAME)
        model.load_state_dict(torch.load(model_path, map_location=Config.DEVICE))
        model.to(Config.DEVICE)
        if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
        
        # OOF (Train)
        # Using val_idx from the MERGED set ensures we get preds for the private part too
        df_val = df_train.iloc[val_idx]
        val_loader = DataLoader(IELTSOrdinalDataset(df_val, tokenizer), batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collator)
        df_train.loc[df_train.index[val_idx], 'deberta_pred'] = predict_fn(model, val_loader, Config.DEVICE)
        
        # Test (Inference)
        test_loader = DataLoader(IELTSOrdinalDataset(df_test, tokenizer), batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collator)
        test_accum.append(predict_fn(model, test_loader, Config.DEVICE))
        
        del model; torch.cuda.empty_cache(); gc.collect()
        
    # Average Test Predictions
    if len(test_accum) > 0:
        df_test['deberta_pred'] = np.mean(test_accum, axis=0)
    else:
        print("⚠️ Warning: No predictions generated (missing models?)")

    # ----------------------------------------------------
    # 3. EXTRACT FEATURES & SAVE
    # ----------------------------------------------------
    tabular_train = extract_features(df_train)
    tabular_test = extract_features(df_test)
    
    # Save
    tabular_train.to_csv(Config.TABULAR_TRAIN_OUT, index=False)
    tabular_test.to_csv(Config.TABULAR_TEST_OUT, index=False)
    
    print("\n✅ DONE!")
    print(f"Train Rows: {len(tabular_train)}, Cols: {len(tabular_train.columns)}")
    print(f"Features Generated: {list(tabular_train.columns)}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import itertools

# INPUT / OUTPUT
TRAIN_FILE = "/kaggle/working/tabular_train_max_features.csv"
TEST_FILE = "/kaggle/working/tabular_test_max_features.csv"
OUTPUT_TRAIN = "tabular_train_enhanced.csv"
OUTPUT_TEST = "tabular_test_enhanced.csv"

def add_advanced_features():
    print("🚀 [0.5/5] Generating Advanced Features (Squared, Interactions, Ratios)...")
    
    df_train = pd.read_csv(TRAIN_FILE)
    df_test = pd.read_csv(TEST_FILE)
    golden_features = [
        'deberta_pred', 'word_count', 'sentence_count', 'paragraph_count', 
        'avg_sent_len', 'unique_words', 'n_complex', 'ttr', 
        'flesch_ease', 'gunning_fog', 'smog'
    ]
    
    # Ensure these cols actually exist
    golden_features = [c for c in golden_features if c in df_train.columns]
    print(f"   ✨ Using {len(golden_features)} Golden Features for interactions.")

    def engineer(df):
        # A. Squared Features (Non-linearity)
        for col in golden_features:
            df[f'{col}_sq'] = df[col] ** 2
            
        # B. Interactions & Ratios (Pairwise)
        # itertools.combinations creates unique pairs (A,B)
        for col1, col2 in itertools.combinations(golden_features, 2):
            # Interaction: A * B
            df[f'{col1}_x_{col2}'] = df[col1] * df[col2]
            
            # Ratio: A / B (Add epsilon to avoid 0 division)
            epsilon = 1e-6
            df[f'{col1}_div_{col2}'] = df[col1] / (df[col2] + epsilon)
            df[f'{col2}_div_{col1}'] = df[col2] / (df[col1] + epsilon)
            
        return df

    # 2. Apply
    df_train = engineer(df_train)
    df_test = engineer(df_test)
    
    # 3. Clean Infinite/NaN values (created by division)
    print("   🧹 Cleaning Infinite/NaN values...")
    df_train = df_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    df_test = df_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # 4. Save
    df_train.to_csv(OUTPUT_TRAIN, index=False)
    df_test.to_csv(OUTPUT_TEST, index=False)
    
    new_cols = len(df_train.columns) - 32 # approx original count
    print(f"   ✅ Added {new_cols} new features!")
    print(f"   📂 Saved to: {OUTPUT_TRAIN}")

if __name__ == "__main__":
    add_advanced_features()

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# CONFIG
TRAIN_FILE = "tabular_train_enhanced.csv"
TEST_FILE = "tabular_test_enhanced.csv"
OUTPUT_TRAIN = "train_with_cls_feat.csv"
OUTPUT_TEST = "test_with_cls_feat.csv"

FOLDS = 5            # User requested 5 Folds
KFOLD_SEED = 42      # Fixed seed for fold splits (Crucial for Stacking alignment)
MODEL_SEEDS = [42, 2024, 777]  # Seeds for Bagging (Averaging)

def generate_lgb_feature():
    print(f"🚀 [1/3] Generating 'lgb_cls_pred' (Bagging {len(MODEL_SEEDS)} Seeds)...")
    
    df = pd.read_csv(TRAIN_FILE)
    test = pd.read_csv(TEST_FILE)
    
    # Ignore standard metadata
    ignore = ['ID', 'Prompt', 'image_description', 'Essay', 'Overall', 'TR', 'CC', 'LR', 'GRA', 'kfold']
    features = [c for c in df.columns if c not in ignore]
    
    y = df['Overall']
    
    # Encode targets (1.0 -> 0, 1.5 -> 1...)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    class_values = le.classes_ 
    
    # Base Params
    base_params = {
        'objective': 'multiclass',
        'num_class': len(class_values),
        'metric': 'multi_logloss',
        'n_estimators': 3000,
        'learning_rate': 0.02,
        'num_leaves': 31,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'device': 'gpu', 
        'verbosity': -1
    }
    
    # We must use the SAME fold split for everything to align later
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=KFOLD_SEED)
    
    oof_preds = np.zeros(len(df))
    test_probs_avg = np.zeros((len(test), len(class_values)))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(df, y_encoded)):
        X_tr, y_tr = df.iloc[tr_idx][features], y_encoded[tr_idx]
        X_val = df.iloc[val_idx][features]
        X_test = test[features]
        
        # --- BAGGING LOOP (Train multiple seeds per fold) ---
        fold_probs = np.zeros((len(X_val), len(class_values)))
        fold_test_probs = np.zeros((len(X_test), len(class_values)))
        
        for seed in MODEL_SEEDS:
            # Update seed
            params = base_params.copy()
            params['random_state'] = seed
            
            clf = lgb.LGBMClassifier(**params)
            clf.fit(X_tr, y_tr)
            
            fold_probs += clf.predict_proba(X_val) / len(MODEL_SEEDS)
            fold_test_probs += clf.predict_proba(X_test) / len(MODEL_SEEDS)
        
        # Convert to Expected Score (Weighted Average)
        oof_preds[val_idx] = np.dot(fold_probs, class_values)
        test_probs_avg += fold_test_probs / FOLDS
        
        print(f"      Fold {fold+1}/{FOLDS} Done.")

    # Final Test Preds
    test_preds = np.dot(test_probs_avg, class_values)
    
    # Add Feature
    df['lgb_cls_pred'] = oof_preds
    test['lgb_cls_pred'] = test_preds
    
    # Save
    df.to_csv(OUTPUT_TRAIN, index=False)
    test.to_csv(OUTPUT_TEST, index=False)
    print(f"   ✅ Saved {OUTPUT_TRAIN} with bagged feature.")

if __name__ == "__main__":
    generate_lgb_feature()

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')

# CONFIG
TRAIN_FILE = "train_with_cls_feat.csv"
TEST_FILE = "test_with_cls_feat.csv"
TEST_ID_FILE = "tabular_test_enhanced.csv"
FOLDS = 5
SEED = 42

class EnhancedPipeline:
    def __init__(self):
        self.weights = []

    def load_data(self):
        print("1. Loading Data...")
        df = pd.read_csv(TRAIN_FILE)
        test = pd.read_csv(TEST_FILE)
        test_ids = pd.read_csv(TEST_ID_FILE)

        # Drop non-feature columns
        ignore = ['ID', 'Prompt', 'image_description', 'Essay', 'Overall', 'TR', 'CC', 'LR', 'GRA', 'kfold', 'id']
        features = [c for c in df.columns if c not in ignore]

        # Simple Imputation (Tree models handle NaNs, but this is safer)
        df[features] = df[features].fillna(df[features].mean())
        test[features] = test[features].fillna(test[features].mean())

        return df, test, test_ids, features

    def train_models(self, df, test, features):
        print("2. Training Models with Early Stopping & OOF...")
        
        y = df['Overall'].values
        # Stratify by converting score to string to treat as classes
        y_strat = df['Overall'].astype(str) 
        X = df[features]
        X_test = test[features]

        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
        
        # Arrays to store predictions
        # Shape: (Rows, 3 models)
        oof_preds = np.zeros((len(X), 3)) 
        test_preds = np.zeros((len(X_test), 3))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat)):
            X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
            X_val, y_val = X.iloc[val_idx], y[val_idx]

            # --- XGBoost ---
            m1 = xgb.XGBRegressor(
                objective='reg:absoluteerror', tree_method='hist', device='cuda',
                n_estimators=2000, learning_rate=0.01, max_depth=6,
                early_stopping_rounds=100, enable_categorical=True
            )
            m1.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
            oof_preds[val_idx, 0] = m1.predict(X_val)
            test_preds[:, 0] += m1.predict(X_test) / FOLDS

            # --- LightGBM ---
            callbacks = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
            m2 = lgb.LGBMRegressor(
                objective='mae', device='gpu', n_estimators=2000, 
                learning_rate=0.01, num_leaves=31, verbosity=-1
            )
            m2.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mae', callbacks=callbacks)
            oof_preds[val_idx, 1] = m2.predict(X_val)
            test_preds[:, 1] += m2.predict(X_test) / FOLDS

            # --- CatBoost ---
            m3 = CatBoostRegressor(
                loss_function='MAE', task_type='GPU', iterations=2000, 
                learning_rate=0.01, depth=6, verbose=0,
                early_stopping_rounds=100
            )
            m3.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            oof_preds[val_idx, 2] = m3.predict(X_val)
            test_preds[:, 2] += m3.predict(X_test) / FOLDS

            print(f"   Fold {fold+1}/{FOLDS} Complete.")

        return oof_preds, test_preds, y

    def optimize_weights(self, oof_preds, y_true):
        print("3. Optimizing Ensemble Weights...")
        
        # Objective function: Minimize MAE
        def loss_func(weights):
            final_prediction = np.average(oof_preds, axis=1, weights=weights)
            return mean_absolute_error(y_true, final_prediction)

        # Constraints: Weights sum to 1, and 0 <= w <= 1
        constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
        bounds = [(0, 1)] * oof_preds.shape[1]
        initial_weights = [1/3, 1/3, 1/3]

        result = minimize(loss_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
        
        self.weights = result.x
        print(f"   📊 Best Weights: XGB={self.weights[0]:.2f}, LGB={self.weights[1]:.2f}, CAT={self.weights[2]:.2f}")
        print(f"   📉 Optimized CV MAE: {result.fun:.4f}")
        
        return self.weights

    def finalize(self, test_preds, test_ids):
        print("4. Generating Submission...")
        
        # Apply optimized weights to test predictions
        final_test_preds = np.average(test_preds, axis=1, weights=self.weights)
        
        # Clip to range
        final_test_preds = np.clip(final_test_preds, 0.5, 9.0)
        
        # Rounding for IELTS (Nearest 0.5)
        final_test_preds = np.round(final_test_preds * 2) / 2
        
        # Save
        id_col = 'id' if 'id' in test_ids.columns else 'ID'
        sub = pd.DataFrame({'id': test_ids[id_col], 'score': final_test_preds})
        
        sub.to_csv("submission_enhanced.csv", index=False)
        print(f"   ✅ Saved 'submission_enhanced.csv'. Mean Score: {sub['score'].mean():.4f}")

def run():
    pipeline = EnhancedPipeline()
    df, test, test_ids, features = pipeline.load_data()
    oof, test_preds, y_true = pipeline.train_models(df, test, features)
    pipeline.optimize_weights(oof, y_true)
    pipeline.finalize(test_preds, test_ids)

if __name__ == "__main__":
    run()